# 11 — Deployment Validation

Final cross-market deployment validation for Project 06.

This notebook is a pure **orchestrator**. For each deployment candidate it
builds the strategy once via the existing pipeline, then hands the resulting
`returns` / `weights` / `equity` to a single src entry point,
`src.analysis.deployment.run_deployment_validation`, which internally composes
`compute_turnover`, `transaction_cost_stress`, `rebalance_analysis`,
`rolling_metrics` and `regime_analysis`. No analytics are implemented here.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[2]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

In [ ]:
import pandas as pd

from src.data.market_configs import MARKET_CONFIGS
from src.data.loader import download_market_data
from src.pipelines.strategy_returns import build_strategy_return_stack
from src.analysis.deployment import run_deployment_validation

## Deployment candidates

Final signal combos selected per market (carried over from notebooks 08/10).

In [ ]:
market_specs = {
    "India": {
        "config": MARKET_CONFIGS["india"],
        "signals": ["mr_ret_10", "low_vol_20", "mr_lowvol_blend"],
    },
    "Brazil": {
        "config": MARKET_CONFIGS["brazil"],
        "signals": ["mom_blend", "mr_lowvol_blend"],
    },
    "Japan": {
        "config": MARKET_CONFIGS["japan"],
        "signals": ["mr_ret_5", "mr_ret_20"],
    },
}

TRANSACTION_COSTS = [0, 2, 5, 10, 20, 50]
REBALANCE_FREQUENCIES = [1, 2, 5, 10, "weekly"]

## Run validation

Build each strategy once (pipeline orchestration), then delegate every
calculation to `run_deployment_validation`.

In [ ]:
validation = {}
cost_tables, rebalance_tables, regime_tables = [], [], []

for market_name, spec in market_specs.items():
    market_data = download_market_data(spec["config"])

    stack = build_strategy_return_stack(
        market_data,
        signal_names=spec["signals"],
        target_vol=0.10,
    )

    returns = stack["dd_vol_ret"]
    weights = (
        stack["panel"]
        .pivot(index="Date", columns="ticker", values="weight")
        .fillna(0)
    )
    equity = (1 + returns).cumprod()

    result = run_deployment_validation(
        returns=returns,
        weights=weights,
        equity=equity,
        transaction_costs=TRANSACTION_COSTS,
        rebalance_frequencies=REBALANCE_FREQUENCIES,
    )

    validation[market_name] = result
    cost_tables.append(result["transaction_cost_stress"].assign(Market=market_name))
    rebalance_tables.append(result["rebalance_analysis"].assign(Market=market_name))
    regime_tables.append(result["regime_analysis"].assign(Market=market_name))

cost_stress_all = pd.concat(cost_tables, ignore_index=True).round(3)
rebalance_all = pd.concat(rebalance_tables, ignore_index=True).round(4)
regime_all = pd.concat(regime_tables, ignore_index=True).round(3)

### Transaction-cost stress (per market)

In [ ]:
cost_stress_all

### Rebalance-frequency turnover (per market)

In [ ]:
rebalance_all

### Drawdown-regime performance (per market)

In [ ]:
regime_all

## Persist results

In [ ]:
results_dir = PROJECT_ROOT / "research/project_06_failure_analysis/results"
results_dir.mkdir(exist_ok=True)

cost_stress_all.to_csv(results_dir / "deployment_validation_cost_stress.csv", index=False)
rebalance_all.to_csv(results_dir / "deployment_validation_rebalance.csv", index=False)
regime_all.to_csv(results_dir / "deployment_validation_regime.csv", index=False)

print("Saved deployment validation results")